# Chapter 9 — Composite Kernels

Reproduces:
- Figure 9.1: Multi-head averaging.
- Figure 9.2: Multi-scale RBF on multi-scale data.
- Figure 9.3: Multi-kernel weights converging during training.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from tabkernels.composite import MultiScale, MultiKernel
from tabkernels.classical import rbf_kernel

torch.manual_seed(42); np.random.seed(42)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

## Figure 9.1: Multi-scale RBF on multi-scale data

In [ ]:
# Two clusters: tight (sigma=0.3) and loose (sigma=1.5).
Xa = torch.randn(20, 2) * 0.3 + torch.tensor([2.0, 0.0])
Xb = torch.randn(20, 2) * 1.5 + torch.tensor([-2.0, 0.0])
X = torch.cat([Xa, Xb])

# Single-bandwidth RBF at three scales.
from tabkernels.classical import rbf_kernel as rbf
scales = [0.3, 1.0, 2.0]
Ws_single = [rbf(X.numpy(), X.numpy(), sigma=s) for s in scales]

# Multi-scale: even mixing weights.
ms = MultiScale(bandwidths=scales, learnable_bandwidths=False)
with torch.no_grad():
    W_ms = ms(X, X)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, W, t in zip(axes, Ws_single + [W_ms.numpy()], [f's={s}' for s in scales] + ['Multi-scale']):
    ax.imshow(W, cmap='viridis')
    ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Figure 9.1: Single-bandwidth RBF at three scales vs multi-scale combination')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_09_01_multiscale.pdf', bbox_inches='tight')
plt.show()

## Figure 9.2: Multi-kernel weights learn during training

In [ ]:
import torch.nn as nn
from tabkernels.core.base import Kernel

class FixedRBFKernel(Kernel):
    def __init__(self, sigma): super().__init__(); self.sigma = sigma
    def forward(self, X1, X2):
        sq = ((X1[:, None] - X2[None, :]) ** 2).sum(-1)
        return torch.exp(-sq / (self.sigma ** 2))

# Generate data where the truth is at sigma=1.0.
X = torch.randn(80, 2)
K_truth = FixedRBFKernel(sigma=1.0)
y_truth = K_truth(X, X)[0, :]  # use row 0 of the 'truth' kernel as target

mk = MultiKernel([
    FixedRBFKernel(sigma=0.3),
    FixedRBFKernel(sigma=1.0),  # truth
    FixedRBFKernel(sigma=3.0),
])
opt = torch.optim.Adam(mk.parameters(), lr=0.1)
weights_history = []
for step in range(200):
    opt.zero_grad()
    K = mk(X, X)
    pred = K[0, :]
    loss = ((pred - y_truth) ** 2).mean()
    loss.backward(); opt.step()
    weights_history.append(mk.weights().numpy())

weights_history = np.array(weights_history)
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for i, s in enumerate([0.3, 1.0, 3.0]):
    ax.plot(weights_history[:, i], label=f's={s}')
ax.set_xlabel('training step'); ax.set_ylabel('mixing weight')
ax.set_title('Figure 9.2: Multi-kernel weights during training (truth: s=1.0)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_09_02_multikernel_learning.pdf', bbox_inches='tight')
plt.show()
print(f'Final weights: {mk.weights().numpy()}')

## Figure 9.3: Multi-head averaging on a multi-modal kernel

In [ ]:
from tabkernels.composite import MultiHead

class _RBFOnSlice(Kernel):
    def __init__(self, slice_idx, sigma):
        super().__init__(); self.slice_idx = slice_idx; self.sigma = sigma
    def forward(self, X1, X2):
        x1s = X1[:, self.slice_idx]; x2s = X2[:, self.slice_idx]
        sq = ((x1s[:, None] - x2s[None, :]) ** 2).sum(-1)
        return torch.exp(-sq / (self.sigma ** 2))

X = torch.randn(15, 4)
heads = [_RBFOnSlice(slice(0, 2), 1.0), _RBFOnSlice(slice(2, 4), 1.0)]
mh = MultiHead(heads)
K_full = mh(X, X)
K_h0 = heads[0](X, X)
K_h1 = heads[1](X, X)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, M, t in zip(axes, [K_h0, K_h1, K_full], ['Head 0 (dims 0-1)', 'Head 1 (dims 2-3)', 'Average (full)']):
    ax.imshow(M, cmap='viridis'); ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Figure 9.3: Multi-head averaging on different feature slices')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_09_03_multihead.pdf', bbox_inches='tight')
plt.show()